# Milestone 3 - Reproduction: BERT-base fine-tuned on SST-2

**Paper:** Devlin et al., *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding* (NAACL 2019)

**Key result to reproduce:** BERT-base on SST-2 reaches **93.5 %** accuracy (paper, Table 1). Our target is about 92 %.

**The paper's fine-tuning grid:** batch size {16, 32} x learning rate {5e-5, 3e-5, 2e-5} x epochs {2, 3, 4}.

**What this notebook does**
1. Trains on the **full** SST-2 training set (67,349 sentences), not the 4,000-sentence subset from Week 8.
2. Reuses our own `SentimentDataset` and `collate_fn` from notebook 02.
3. Reruns the PyTorch checklist items (first loss close to ln(C), non-zero gradients on every layer).
4. Runs the 6-configuration grid (batch x learning rate) at 3 epochs, then an epoch sweep (2 and 4) and a seed check on the best configuration.
5. Puts every run in one results table, plots train and validation curves for every run, and compares our number with the paper's.
6. Saves results and checkpoints to Google Drive. Finished runs are skipped when the notebook is re-run, so a Colab disconnect does not lose work.

**Evaluation split:** the GLUE test labels are hidden (-1), so we evaluate on the 872-sentence **validation** set. The paper's 93.5 % is from the GLUE test server, which matters when explaining the gap (see section 7).

## 0. Setup

In [ ]:
!pip -q install transformers datasets accelerate

In [ ]:
import os, json, time, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# Save everything to Drive (Colab disconnects sometimes, so checkpoints must live on Drive)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/BERT_M3"
except ImportError:
    OUT_DIR = "./BERT_M3"
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(OUT_DIR, "results.json")
BEST_PATH = os.path.join(OUT_DIR, "best_model.pt")

SEED = 42
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
EPOCHS = 3            # grid runs use 3 epochs (2 and 4 are tested later on the best config)
WARMUP_FRAC = 0.1     # linear warm-up for the first 10 % of steps, then linear decay (as in the BERT paper)
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
PAPER_ACC = 93.5      # BERT-base, SST-2, Table 1 of the paper (GLUE test server)
GRID_BATCH = [16, 32]
GRID_LR = [5e-5, 3e-5, 2e-5]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"   # mixed precision (fp16) on the T4

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("PyTorch:", torch.__version__, "| Device:", device, "| Mixed precision:", USE_AMP)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Outputs saved to:", OUT_DIR)

## 1. Data: our own Dataset and DataLoader (from notebook 02)

- `SentimentDataset` tokenises each sentence once, with no padding.
- `collate_fn` pads each batch to its longest sentence (dynamic padding) and builds the `attention_mask`.
- Only the train and validation splits are used (test labels are hidden).
- Normalisation rule for text: any statistic must come from the training split only. Here the tokenizer vocabulary is fixed by the pretrained model, so nothing is fitted on validation data.

In [ ]:
raw = load_dataset("nyu-mll/glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_ID = tokenizer.pad_token_id

class SentimentDataset(Dataset):
    def __init__(self, hf_split, tokenizer, text_col="sentence", label_col="label", max_length=MAX_LEN):
        texts = list(hf_split[text_col])
        self.texts = texts
        self.labels = list(hf_split[label_col])
        enc = tokenizer(texts, truncation=True, max_length=max_length)   # no padding here
        self.input_ids = enc["input_ids"]
        self.token_type_ids = enc["token_type_ids"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx],
                "token_type_ids": self.token_type_ids[idx],
                "label": self.labels[idx]}

def collate_fn(batch):
    B = len(batch)
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids      = torch.full((B, max_len), PAD_ID, dtype=torch.long)
    token_type_ids = torch.zeros((B, max_len), dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["input_ids"])
        input_ids[i, :n]      = torch.tensor(x["input_ids"], dtype=torch.long)
        token_type_ids[i, :n] = torch.tensor(x["token_type_ids"], dtype=torch.long)
        attention_mask[i, :n] = 1
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
    return {"input_ids": input_ids, "token_type_ids": token_type_ids,
            "attention_mask": attention_mask, "labels": labels}

train_ds = SentimentDataset(raw["train"], tokenizer)
val_ds   = SentimentDataset(raw["validation"], tokenizer)
print("Train size:", len(train_ds), "| Validation size:", len(val_ds))
print("Train labels:", dict(Counter(train_ds.labels)), "| Val labels:", dict(Counter(val_ds.labels)))

def make_loaders(batch_size, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, generator=g)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)
    return train_loader, val_loader

# Check one batch (shapes, dtypes, padding)
tl, vl = make_loaders(32, SEED)
b = next(iter(tl))
for k, v in b.items():
    print(f"{k:16s} shape={tuple(v.shape)} dtype={v.dtype}")
assert torch.equal(b["attention_mask"] == 0, b["input_ids"] == PAD_ID)
assert (b["input_ids"][:, 0] == tokenizer.cls_token_id).all()
print("Batch checks passed.")

## 2. PyTorch checklist re-check on the full pipeline

Two sanity checks:
- The first loss of an untrained classification head should be close to ln(C) = ln(2) = 0.693 for two classes.
- Every layer should receive non-zero gradients.

*Note:* the key-projection **bias** inside self-attention (`attention.self.key.bias`) always has a gradient of exactly zero in theory. Adding a constant to all keys shifts every attention score of a query by the same amount, and softmax ignores that shift. So that one is not a bug and is excluded from the check.

In [ ]:
def build_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

set_seed(SEED)
train_loader, val_loader = make_loaders(32, SEED)
model = build_model()

# 1) First loss close to ln(C)
model.eval()
losses = []
with torch.no_grad():
    for i, bt in enumerate(train_loader):
        if i == 20:
            break
        bt = {k: v.to(device) for k, v in bt.items()}
        losses.append(model(**bt).loss.item())
first_loss = float(np.mean(losses))
print(f"Mean loss of the untrained head over 20 batches: {first_loss:.4f} | ln(2) = {math.log(2):.4f}")
assert abs(first_loss - math.log(2)) < 0.1, "first loss is far from ln(C)"

# 2) Non-zero gradients on every layer
model.train()
model.zero_grad()
bt = {k: v.to(device) for k, v in next(iter(train_loader)).items()}
model(**bt).loss.backward()

zero_grad, total = [], 0
for name, p in model.named_parameters():
    total += 1
    if p.grad is None or p.grad.abs().max().item() == 0.0:
        zero_grad.append(name)
unexpected = [n for n in zero_grad if not n.endswith("attention.self.key.bias")]
print(f"Parameter tensors: {total} | with all-zero gradient: {len(zero_grad)} "
      f"(expected: the 12 key biases) | unexpected: {len(unexpected)}")
assert not unexpected, unexpected

show = ["bert.embeddings.word_embeddings.weight",
        "bert.encoder.layer.0.attention.self.query.weight",
        "bert.encoder.layer.5.intermediate.dense.weight",
        "bert.encoder.layer.11.output.dense.weight",
        "bert.pooler.dense.weight",
        "classifier.weight"]
params = dict(model.named_parameters())
print("\nGradient norms at selected layers:")
for n in show:
    print(f"  {n:55s} {params[n].grad.norm().item():.6f}")
print("Sanity checks passed.")

del model
torch.cuda.empty_cache()

## 3. Training and evaluation code

- AdamW with weight decay 0.01 (not applied to biases and LayerNorm), linear warm-up for 10 % of steps, then linear decay to 0, gradient clipping at 1.0.
- Mixed precision (fp16 autocast + GradScaler) so each run is fast on the T4.
- `model.train()` and `model.eval()` are set correctly, `zero_grad` is called every step, and the seed is fixed per run.
- After each epoch the run saves a resume checkpoint to Drive. If Colab disconnects, re-running the cell continues from the last finished epoch. Runs that are already finished are skipped.
- Only the best final model is kept on Drive (`best_model.pt`) to save space.

In [ ]:
def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            return json.load(f)
    return {}

def save_results(res):
    with open(RESULTS_PATH, "w") as f:
        json.dump(res, f, indent=2)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for bt in loader:
        bt = {k: v.to(device) for k, v in bt.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(**bt)
        bs = bt["labels"].size(0)
        tot_loss += out.loss.item() * bs
        correct += (out.logits.argmax(dim=1) == bt["labels"]).sum().item()
        n += bs
    return tot_loss / n, correct / n

def run_experiment(batch_size, lr, epochs=EPOCHS, seed=SEED, tag="grid"):
    key = f"bs{batch_size}_lr{lr:g}_ep{epochs}_s{seed}"
    results = load_results()
    if key in results:
        r = results[key]
        print(f"[skip] {key} already finished: final val acc = {r['final_val_acc']*100:.2f}%")
        return r

    print(f"\n=== {key} ===")
    set_seed(seed)
    train_loader, val_loader = make_loaders(batch_size, seed)
    model = build_model()

    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(groups, lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    hist = {"train_loss": [], "val_loss": [], "val_acc": [], "epoch_time_s": []}
    start_epoch = 0
    resume_path = os.path.join(OUT_DIR, f"resume_{key}.pt")
    if os.path.exists(resume_path):
        ck = torch.load(resume_path, map_location=device)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
        hist, start_epoch = ck["hist"], ck["epoch"]
        print(f"Resumed from checkpoint after epoch {start_epoch}")

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        train_loader.generator.manual_seed(seed + epoch)   # reproducible shuffle (also correct after a resume)
        model.train()
        run_loss, seen = 0.0, 0
        for bt in tqdm(train_loader, desc=f"epoch {epoch+1}/{epochs}", leave=False):
            bt = {k: v.to(device) for k, v in bt.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(**bt)
            scaler.scale(out.loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            bs = bt["labels"].size(0)
            run_loss += out.loss.item() * bs
            seen += bs
        val_loss, val_acc = evaluate(model, val_loader)
        hist["train_loss"].append(run_loss / seen)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)
        hist["epoch_time_s"].append(time.time() - t0)
        print(f"epoch {epoch+1}: train loss {run_loss/seen:.4f} | val loss {val_loss:.4f} | "
              f"val acc {val_acc*100:.2f}% | {time.time()-t0:.0f}s")
        if epoch + 1 < epochs:   # resume checkpoint on Drive (overwritten every epoch)
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                        "hist": hist, "epoch": epoch + 1}, resume_path)

    result = {"tag": tag, "batch_size": batch_size, "lr": lr, "epochs": epochs, "seed": seed,
              "history": hist, "final_val_acc": hist["val_acc"][-1],
              "best_val_acc": max(hist["val_acc"]), "best_epoch": int(np.argmax(hist["val_acc"])) + 1,
              "train_minutes": sum(hist["epoch_time_s"]) / 60}

    prev_best = max([r["final_val_acc"] for r in results.values()], default=-1.0)
    if result["final_val_acc"] > prev_best:   # keep only the best final model on Drive
        torch.save({"state_dict": model.state_dict(), "key": key, "val_acc": result["final_val_acc"]}, BEST_PATH)
        print("New best model saved to Drive.")
    results[key] = result
    save_results(results)
    if os.path.exists(resume_path):
        os.remove(resume_path)

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    return result

## 4. The grid: 2 batch sizes x 3 learning rates (3 epochs each)

Same seed (42), same split and same schedule for every run, so the only thing that changes between runs is the batch size or the learning rate. Expect roughly 1 to 1.5 hours on a T4. If Colab disconnects, just reconnect, rerun the setup, data and function cells, and run this cell again. Finished runs are skipped.

In [ ]:
for bs in GRID_BATCH:
    for lr in GRID_LR:
        run_experiment(bs, lr, epochs=EPOCHS, seed=SEED, tag="grid")

## 5. Results table

In [ ]:
def results_df(tag):
    rows = []
    for k, r in load_results().items():
        if r["tag"] != tag:
            continue
        row = {"batch_size": r["batch_size"], "lr": r["lr"], "epochs": r["epochs"], "seed": r["seed"]}
        for i, a in enumerate(r["history"]["val_acc"], 1):
            row[f"val_acc_ep{i} (%)"] = round(a * 100, 2)
        row["final val acc (%)"] = round(r["final_val_acc"] * 100, 2)
        row["final val loss"] = round(r["history"]["val_loss"][-1], 4)
        row["final train loss"] = round(r["history"]["train_loss"][-1], 4)
        row["time (min)"] = round(r["train_minutes"], 1)
        rows.append(row)
    return pd.DataFrame(rows)

grid_df = results_df("grid").sort_values("final val acc (%)", ascending=False).reset_index(drop=True)
grid_df.to_csv(os.path.join(OUT_DIR, "grid_results.csv"), index=False)
display(grid_df)

best = grid_df.iloc[0]
BEST_BS, BEST_LR = int(best["batch_size"]), float(best["lr"])
print(f"Best configuration: batch size {BEST_BS}, lr {BEST_LR:g}, {EPOCHS} epochs -> "
      f"{best['final val acc (%)']:.2f}% validation accuracy")

### Train and validation curves for every run

In [ ]:
res = load_results()
grid_keys = [k for k, r in res.items() if r["tag"] == "grid"]
grid_keys.sort(key=lambda k: (res[k]["batch_size"], -res[k]["lr"]))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, k in zip(axes.ravel(), grid_keys):
    h = res[k]["history"]
    ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], marker="o", label="train loss")
    ax.plot(ep, h["val_loss"], marker="o", label="validation loss")
    ax.set_title(f"batch {res[k]['batch_size']}, lr {res[k]['lr']:g}  (val acc {res[k]['final_val_acc']*100:.2f}%)")
    ax.set_xlabel("Epoch"); ax.set_xticks(list(ep)); ax.grid(True)
axes[0, 0].set_ylabel("Loss"); axes[1, 0].set_ylabel("Loss"); axes[0, 0].legend()
plt.suptitle("BERT-base on SST-2: train and validation loss for every grid run", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "grid_loss_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
for k in grid_keys:
    h = res[k]["history"]
    plt.plot(range(1, len(h["val_acc"]) + 1), [a * 100 for a in h["val_acc"]], marker="o",
             label=f"bs{res[k]['batch_size']}, lr {res[k]['lr']:g}")
plt.axhline(PAPER_ACC, color="red", linestyle="--", label=f"paper ({PAPER_ACC}%)")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy (%)"); plt.xticks(range(1, EPOCHS + 1))
plt.title("Validation accuracy per epoch, all grid runs"); plt.legend(fontsize=8); plt.grid(True)
plt.savefig(os.path.join(OUT_DIR, "grid_val_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Completing the paper's grid: epochs and seeds on the best configuration

- **Epoch sweep:** the paper tries 2, 3 and 4 epochs. The learning-rate schedule depends on the total number of epochs, so 2 and 4 are separate runs on the best (batch size, learning rate). The 3-epoch run is already in the grid.
- **Seed check:** the validation set has only 872 sentences, so one sentence is about 0.11 %. Running the best configuration with two more seeds gives a mean and a spread, which is an honest way to report the number.

Set `RUN_EPOCH_SWEEP` or `RUN_SEED_CHECK` to `False` to skip either one if time is short.

In [ ]:
RUN_EPOCH_SWEEP = True
RUN_SEED_CHECK = True

if RUN_EPOCH_SWEEP:
    for e in [2, 4]:
        run_experiment(BEST_BS, BEST_LR, epochs=e, seed=SEED, tag="epochs")

if RUN_SEED_CHECK:
    for s in [1, 2]:
        run_experiment(BEST_BS, BEST_LR, epochs=EPOCHS, seed=s, tag="seeds")

# Epoch table (2, 3, 4 epochs at the best batch size and learning rate)
ep_rows = []
for k, r in load_results().items():
    if r["batch_size"] == BEST_BS and r["lr"] == BEST_LR and r["seed"] == SEED and r["tag"] in ("grid", "epochs"):
        ep_rows.append({"epochs": r["epochs"], "final val acc (%)": round(r["final_val_acc"] * 100, 2),
                        "final val loss": round(r["history"]["val_loss"][-1], 4)})
epoch_df = pd.DataFrame(ep_rows).sort_values("epochs").reset_index(drop=True)
print(f"Epoch sweep (batch {BEST_BS}, lr {BEST_LR:g}, seed {SEED}):")
display(epoch_df)

# Seed table (3 seeds: 42, 1, 2)
seed_accs = [r["final_val_acc"] * 100 for r in load_results().values()
             if r["batch_size"] == BEST_BS and r["lr"] == BEST_LR and r["epochs"] == EPOCHS
             and r["tag"] in ("grid", "seeds")]
seed_mean, seed_std = float(np.mean(seed_accs)), float(np.std(seed_accs))
print(f"Best config over {len(seed_accs)} seeds: {seed_mean:.2f}% +/- {seed_std:.2f}  (values: "
      + ", ".join(f"{a:.2f}" for a in seed_accs) + ")")

## 7. Our number next to the paper's

In [ ]:
best_acc = float(grid_df.iloc[0]["final val acc (%)"])
gap = PAPER_ACC - best_acc
gap_mean = PAPER_ACC - seed_mean

comparison = pd.DataFrame([
    {"Source": "BERT paper (BERT-base, GLUE test server)", "SST-2 accuracy (%)": PAPER_ACC},
    {"Source": f"Ours, best grid run (bs {BEST_BS}, lr {BEST_LR:g}, {EPOCHS} ep), validation set", "SST-2 accuracy (%)": best_acc},
    {"Source": f"Ours, same config, mean of {len(seed_accs)} seeds, validation set", "SST-2 accuracy (%)": round(seed_mean, 2)},
])
display(comparison)
print(f"Gap to the paper: {gap:.2f} points (best run), {gap_mean:.2f} points (seed mean).")
print(f"Previous Week-8 baseline (4,000 sentences, 2 epochs): 89.11% -> full data gained {best_acc - 89.11:.2f} points.")

plt.figure(figsize=(7, 4))
labels = ["Paper", "Ours (best run)", f"Ours (mean of {len(seed_accs)} seeds)"]
vals = [PAPER_ACC, best_acc, seed_mean]
plt.bar(labels, vals)
plt.ylim(min(vals) - 3, 95)
for i, v in enumerate(vals):
    plt.text(i, v + 0.1, f"{v:.2f}", ha="center")
plt.ylabel("SST-2 accuracy (%)"); plt.title("Reproduction: BERT-base on SST-2")
plt.savefig(os.path.join(OUT_DIR, "paper_vs_ours.png"), dpi=150, bbox_inches="tight")
plt.show()

### Why our number differs from the paper's (the gap explained)

1. **Different evaluation set.** The paper reports the GLUE **test** set (scored on the GLUE server). The test labels are hidden, so we evaluate on the 872-sentence **validation** set. These are different sentences, and a gap of 0.5 to 1 point is common between them.
2. **Small validation set and selection effect.** One sentence is about 0.11 %, so the seed-to-seed spread (section 6) is comparable to the differences between several grid configurations. We also pick the best configuration on the same validation set we report, which makes the best-run number slightly optimistic. The seed mean is the fairer figure.
3. **Different implementation details.** We use the Hugging Face PyTorch port of `bert-base-uncased` with mixed-precision (fp16) training, a fixed seed and a single run per setting. The paper used the original TensorFlow code and checkpoints, and it chose the best setting on the dev set among its grid.
4. **Training data format.** GLUE's SST-2 training set includes phrase-level examples (67,349 items), while validation and test contain only full sentences. This affects how closely the training distribution matches the evaluation distribution.

Taken together, a result within about 1 point of 93.5 % at Colab scale counts as a successful reproduction. The remaining gap is explained by points 1 to 3, not by a bug (the sanity checks in section 2 passed, and the overfit-one-batch test from Week 8 reached near-zero loss).

## 8. Error analysis starter (feeds the report and M4)

Load the best saved model from Drive, show the confusion matrix on the validation set, and list some mistakes.

In [ ]:
ck = torch.load(BEST_PATH, map_location=device)
print("Best saved model:", ck["key"], "| val acc:", round(ck["val_acc"] * 100, 2), "%")
model = build_model()
model.load_state_dict(ck["state_dict"])
model.eval()

_, vl = make_loaders(32, SEED)
preds, golds = [], []
with torch.no_grad():
    for bt in vl:
        bt = {k: v.to(device) for k, v in bt.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(**bt)
        preds += out.logits.argmax(dim=1).cpu().tolist()
        golds += bt["labels"].cpu().tolist()
preds, golds = np.array(preds), np.array(golds)

cm = np.zeros((2, 2), dtype=int)
for g_, p_ in zip(golds, preds):
    cm[g_, p_] += 1
print("Confusion matrix (rows = true, columns = predicted)")
print(pd.DataFrame(cm, index=["true Neg", "true Pos"], columns=["pred Neg", "pred Pos"]))
print(f"Accuracy: {(preds == golds).mean()*100:.2f}%")

names = {0: "Negative", 1: "Positive"}
wrong = np.where(preds != golds)[0]
print(f"\n{len(wrong)} mistakes. First 10:")
for i in wrong[:10]:
    print(f"- true {names[golds[i]]:8s} | predicted {names[preds[i]]:8s} | {val_ds.texts[i]}")

## 9. M3 summary to show in the lab

In [ ]:
print("M3 - REPRODUCTION SUMMARY")
print("-" * 60)
print(f"Task: BERT-base-uncased fine-tuned on SST-2 (full train set, {len(train_ds):,} sentences)")
print(f"Grid: batch {GRID_BATCH} x lr {GRID_LR}, {EPOCHS} epochs, seed {SEED}, fp16, AdamW + linear warm-up/decay")
print(f"Best grid run: batch {BEST_BS}, lr {BEST_LR:g} -> {best_acc:.2f}% validation accuracy")
print(f"Best config over {len(seed_accs)} seeds: {seed_mean:.2f}% +/- {seed_std:.2f}")
print(f"Paper: {PAPER_ACC}% (GLUE test) | gap: {gap:.2f} (best run), {gap_mean:.2f} (seed mean)")
print("Epoch sweep at the best config:", ", ".join(f"{int(r.epochs)} ep = {r['final val acc (%)']:.2f}%" for _, r in epoch_df.iterrows()))
print(f"All results, curves and the best checkpoint are in: {OUT_DIR}")